# 20 Data_Cleaning_Filtering_And_Dedup

## Problem

原始抓取数据为什么不能直接拿去预训练？数据清洗、质量过滤、去重和近重复检测分别在解决什么问题？

## Dependencies

建议先完成 `19_data_collection_and_crawling.ipynb`，因为清洗和过滤的对象就是上一阶段存下来的原始记录。


## Module_Goals

- 理解为什么抓取数据通常伴随大量噪声
- 理解 HTML 清理、语言识别、长度过滤、质量过滤的基本逻辑
- 理解完全重复与近重复数据的区别
- 能写一个最小清洗、过滤和去重流程示意

## Scope_And_Approach

这个 notebook 关注的是从原始记录到训练文本的最后一公里。重点不是做一个工业级数据平台，而是说明为什么数据质量会直接决定模型底座质量，以及最常见的清洗步骤具体在干什么。


## Context_And_Motivation

原始网页和文档数据常见的问题包括：

- HTML 标签、脚本、导航栏、cookie 弹窗混入正文
- 站点模板导致大量重复段落
- 极短文本、乱码文本、广告文本、拼接错误文本
- 多语言混杂或语言识别错误
- 同一篇文章被镜像、转载、改格式后重复出现

如果这些问题不处理，模型会把大量无效模式也当作语言规律学进去，最后直接影响训练效率和输出质量。


In [ ]:
raw_texts = [
    '<html><body>Buy now!<div>Deep learning tutorial</div></body></html>',
    'Deep learning tutorial',
    'Deep learning tutorial!!!',
    'Deep learning tutorial for beginners',
    'ok',
]

def normalize_text(text):
    text = text.replace('<html>', '').replace('</html>', '')
    text = text.replace('<body>', '').replace('</body>', '')
    text = text.replace('<div>', ' ').replace('</div>', ' ')
    text = text.replace('!!!', '!')
    return ' '.join(text.split()).strip()

normalized = [normalize_text(text) for text in raw_texts]
print('normalized =', normalized)


In [ ]:
def simple_quality_filter(text):
    if len(text) < 10:
        return False
    if text.lower().count('buy now') > 0:
        return False
    return True

filtered = [text for text in normalized if simple_quality_filter(text)]
exact_deduped = list(dict.fromkeys(filtered))

print('filtered =', filtered)
print('exact_deduped =', exact_deduped)


In [ ]:
def jaccard_similarity(a, b):
    a_tokens = set(a.lower().split())
    b_tokens = set(b.lower().split())
    return len(a_tokens & b_tokens) / max(1, len(a_tokens | b_tokens))

near_dup_threshold = 0.75
pairs = []
for i in range(len(exact_deduped)):
    for j in range(i + 1, len(exact_deduped)):
        sim = jaccard_similarity(exact_deduped[i], exact_deduped[j])
        pairs.append((exact_deduped[i], exact_deduped[j], round(sim, 4), sim >= near_dup_threshold))

for left, right, sim, is_near_dup in pairs:
    print(f'similarity={sim} | near_dup={is_near_dup} | {left!r} <-> {right!r}')


In [ ]:
# 一个最小质量分桶示意
def quality_bucket(text):
    if len(text) > 30 and 'tutorial' in text.lower():
        return 'high_quality'
    if len(text) > 12:
        return 'medium_quality'
    return 'review_or_drop'

bucketed = [(text, quality_bucket(text)) for text in exact_deduped]
for item in bucketed:
    print(item)


## Implementation_Notes

一个更完整的数据清洗流程通常包括：

1. 正文清理：去掉标签、脚本、模板噪声。
2. 基础过滤：长度过短、字符分布异常、乱码、广告化文本先排掉。
3. 语言与来源校验：保证同一数据桶内部的分布可控。
4. 完全去重：完全相同文本只保留一份。
5. 近重复检测：相似模板页、镜像页、转载页做相似度去重。
6. 质量分层：高质量语料、普通语料、待人工复查语料分桶。

如果数据量继续变大，近重复检测通常会从简单 token overlap 升级到 shingling、MinHash、LSH 这一类更适合大规模去重的方案。

## Trade_Offs_And_Risk_Points

- 过滤太松，模型会学到大量模板噪声和低质量模式。
- 过滤太严，可能误删有价值的长尾知识或稀缺领域内容。
- 只做完全去重，不做近重复检测，训练集仍可能被镜像页和转载页污染。
- 只做文本层清洗，不保留来源信息，后面很难定位问题数据桶。
- 不做质量分桶，会让高价值语料和低价值语料混在同一训练阶段里。

## Checkpoints

- 解释为什么“完全重复”和“近重复”是两个不同问题。
- 解释为什么数据清洗不是一次正则替换就结束。
- 说明为什么数据质量会直接影响预训练收益。
- 说明为什么质量分桶对于多阶段训练有意义。
- 解释 near-duplicate threshold 调高或调低会带来什么影响。

## Summary

数据清洗和去重解决的是“原材料能不能真正进入模型”的问题。模型能力不只取决于参数规模和训练时长，也取决于输入语料是不是足够干净、足够多样、足够高质量。对大模型来说，数据工程并不是辅助项，而是底座的一部分。

## Next_Module

到这里，项目主线已经把结构、训练和数据三部分连起来了。后续如果继续扩展，可以把采样、量化、长上下文和推理服务进一步接上。
